<a href="https://colab.research.google.com/github/khalisazalfasalsabila/Re202-pemrograma_53/blob/main/new_smart_trash_sorting_khalisa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import customtkinter as ctk
import random
import tkinter as tk
import math
import time

# --- KONEKSI SERIAL ARDUINO ---
try:
    import serial
    # Ganti 'COM3' sesuai dengan port Arduino Anda yang terbaca di Arduino IDE
    ser = serial.Serial('COM3', 9600, timeout=1)
    ARDUINO_TERHUBUNG = True
except Exception as e:
    ARDUINO_TERHUBUNG = False

try:
    import cv2
    from PIL import Image, ImageTk
    OPENCV_TERSEDIA = True
except ImportError:
    OPENCV_TERSEDIA = False

ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("blue")

class SistemPemilahSampahPintar(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.ESP32_CAM_URL = "http://192.168.1.100/stream"
        self.cap = None
        self.running_camera = False

        # Palette Warna Visual
        self.BG_UTAMA = "#090D16"
        self.BG_PANEL = "#111827"
        self.AKSEN_CYAN = "#00F0FF"
        self.STATUS_HIJAU = "#39FF14"
        self.ALARM_MERAH = "#FF0055"
        self.TEKS_PUTIH = "#E2E8F0"
        self.FONT_CODE = "Consolas"

        self.title("SMART TRASH SORTING BIN - ARDUINO HARDWARE LINK")
        self.geometry("1250x750")
        self.resizable(False, False)
        self.configure(fg_color=self.BG_UTAMA)

        self.animasi_id = None
        self.is_otomatis = True
        self.KAPASITAS_MAKS = 10  # Batas maksimal sampah di sini
        self.kedip_status = False
        self.waktu_start = time.time()
        self.kategori_aktif = "KOSONG"

        # Grid Layout Utama
        self.main_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.main_frame.pack(fill="both", expand=True, padx=20, pady=(20, 10))
        self.main_frame.grid_columnconfigure(0, weight=0, minsize=360)
        self.main_frame.grid_columnconfigure(1, weight=1)
        self.main_frame.grid_rowconfigure(0, weight=1)

        # =================================================================
        # PANEL SEKTOR KIRI
        # =================================================================
        self.frame_kiri = ctk.CTkFrame(self.main_frame, fg_color="transparent")
        self.frame_kiri.grid(row=0, column=0, padx=(0, 15), sticky="nsew")

        self.panel_kamera = ctk.CTkFrame(self.frame_kiri, fg_color=self.BG_PANEL, corner_radius=4, border_width=1, border_color="#1E293B")
        self.panel_kamera.pack(fill="x", pady=(0, 15))

        status_cam_txt = "● ESP32-CAM LIVE" if OPENCV_TERSEDIA else "○ ESP32-CAM (SIMULATOR)"
        status_cam_clr = self.STATUS_HIJAU if OPENCV_TERSEDIA else "#FF9900"
        ctk.CTkLabel(self.panel_kamera, text=status_cam_txt, font=(self.FONT_CODE, 11, "bold"), text_color=status_cam_clr).pack(pady=(8, 2), anchor="w", padx=15)
        self.lbl_video = ctk.CTkLabel(self.panel_kamera, text="MENGHUBUNGKAN KAMERA...", fg_color="#060911", height=200)
        self.lbl_video.pack(fill="x", padx=15, pady=(5, 10))

        self.panel_remote = ctk.CTkFrame(self.frame_kiri, fg_color=self.BG_PANEL, corner_radius=4, border_width=1, border_color="#1E293B")
        self.panel_remote.pack(fill="both", expand=True)

        status_hw = "● ARDUINO CONNECTED" if ARDUINO_TERHUBUNG else "○ ARDUINO DISCONNECTED (SIM MODE)"
        clr_hw = self.STATUS_HIJAU if ARDUINO_TERHUBUNG else self.ALARM_MERAH
        ctk.CTkLabel(self.panel_remote, text=status_hw, font=(self.FONT_CODE, 10, "bold"), text_color=clr_hw).pack(anchor="w", padx=15, pady=(5,0))

        ctk.CTkLabel(self.panel_remote, text="REMOTE FLIP CONTROL", font=(self.FONT_CODE, 13, "bold"), text_color=self.AKSEN_CYAN).pack(pady=(10, 5), anchor="w", padx=15)

        self.btn_mode_auto = ctk.CTkButton(self.panel_remote, text="● SYSTEM AUTONOMOUS", fg_color="#10B981", text_color="#FFFFFF", font=(self.FONT_CODE, 12, "bold"), command=self.set_mode_autonomous)
        self.btn_mode_auto.pack(fill="x", padx=15, pady=2)

        self.btn_mode_rc = ctk.CTkButton(self.panel_remote, text="○ MANUAL OVERRIDE (RC)", fg_color="#374151", text_color="#9CA3AF", font=(self.FONT_CODE, 12, "bold"), command=self.set_mode_remote)
        self.btn_mode_rc.pack(fill="x", padx=15, pady=2)

        self.lbl_status_aktif = ctk.CTkLabel(self.panel_remote, text="AKTIF: AUTO", font=(self.FONT_CODE, 11, "bold"), text_color=self.STATUS_HIJAU)
        self.lbl_status_aktif.pack(pady=5)

        self.grid_dpad = ctk.CTkFrame(self.panel_remote, fg_color="transparent")
        self.grid_dpad.pack(pady=15)

        self.btn_flip_logam = ctk.CTkButton(self.grid_dpad, text="◀ LOGAM\n(Flip Kiri)", width=110, height=60, fg_color="#1F2937", font=(self.FONT_CODE, 11, "bold"), command=lambda: self.eksekusi_flip_manual("LOGAM"))
        self.btn_flip_logam.grid(row=0, column=0, padx=10, pady=5)

        self.btn_flip_center = ctk.CTkButton(self.grid_dpad, text="■ RESET\n(Center 90°)", width=90, height=60, fg_color="#1F2937", font=(self.FONT_CODE, 11, "bold"), command=lambda: self.eksekusi_flip_manual("CENTER"))
        self.btn_flip_center.grid(row=0, column=1, padx=5, pady=5)

        self.btn_flip_nonlogam = ctk.CTkButton(self.grid_dpad, text="NON-LOGAM ▶\n(Flip Kanan)", width=110, height=60, fg_color="#1F2937", font=(self.FONT_CODE, 11, "bold"), command=lambda: self.eksekusi_flip_manual("NON-LOGAM"))
        self.btn_flip_nonlogam.grid(row=0, column=2, padx=10, pady=5)

        self.list_dpad = [self.btn_flip_logam, self.btn_flip_center, self.btn_flip_nonlogam]

        self.panel_keyboard = ctk.CTkFrame(self.panel_remote, fg_color="#060911", corner_radius=2, border_width=1, border_color="#1E293B")
        self.panel_keyboard.pack(fill="x", padx=15, pady=(5, 10))
        txt_shortcut = "Tombol A / ←  : Flip Kiri (LOGAM)\nTombol D / →  : Flip Kanan (NON-LOGAM)\nTombol S / Space: Kembalikan Posisi Tengah"
        ctk.CTkLabel(self.panel_keyboard, text=txt_shortcut, font=(self.FONT_CODE, 10), text_color=self.AKSEN_CYAN, justify="left").pack(anchor="w", padx=10, pady=5)

        self.bind("<KeyPress>", self.logika_input_keyboard)

        # =================================================================
        # PANEL SEKTOR KANAN (UPDATE DATA BOLA UTAMA)
        # =================================================================
        self.frame_kanan = ctk.CTkFrame(self.main_frame, fg_color="transparent")
        self.frame_kanan.grid(row=0, column=1, sticky="nsew")

        # Top Counter Panel
        self.panel_sensor = ctk.CTkFrame(self.frame_kanan, fg_color=self.BG_PANEL, corner_radius=4, border_width=1, border_color="#1E293B")
        self.panel_sensor.pack(fill="x", pady=(0, 15))

        self.grid_tabel = ctk.CTkFrame(self.panel_sensor, fg_color="transparent")
        self.grid_tabel.pack(fill="x", padx=15, pady=8)
        self.grid_tabel.grid_columnconfigure((0,1,2,3,4,5), weight=1)

        ctk.CTkLabel(self.grid_tabel, text="Data Logam:", font=(self.FONT_CODE, 11)).grid(row=0, column=0, sticky="e")
        self.txt_logam = ctk.CTkEntry(self.grid_tabel, width=60, height=25, justify="center", fg_color="#090D16", text_color=self.STATUS_HIJAU)
        self.txt_logam.insert(0, "0")
        self.txt_logam.grid(row=0, column=1, sticky="w")

        ctk.CTkLabel(self.grid_tabel, text="Non-Logam:", font=(self.FONT_CODE, 11)).grid(row=0, column=2, sticky="e")
        self.txt_non_logam = ctk.CTkEntry(self.grid_tabel, width=60, height=25, justify="center", fg_color="#090D16", text_color=self.STATUS_HIJAU)
        self.txt_non_logam.insert(0, "0")
        self.txt_non_logam.grid(row=0, column=3, sticky="w")

        ctk.CTkLabel(self.grid_tabel, text="Sudut Servo:", font=(self.FONT_CODE, 11)).grid(row=0, column=4, sticky="e")
        self.txt_servo = ctk.CTkEntry(self.grid_tabel, width=60, height=25, justify="center", fg_color="#090D16", text_color=self.STATUS_HIJAU)
        self.txt_servo.insert(0, "90")
        self.txt_servo.grid(row=0, column=5, sticky="w")

        # Canvas Animasi
        self.panel_canvas = ctk.CTkFrame(self.frame_kanan, fg_color=self.BG_PANEL, corner_radius=4, border_width=1, border_color="#1E293B")
        self.panel_canvas.pack(fill="both", expand=True, pady=(0, 15))

        self.canvas_monitor = tk.Canvas(self.panel_canvas, width=700, height=230, bg="#060911", highlightthickness=0)
        self.canvas_monitor.pack(pady=10, padx=15, expand=True)
        self.center_x = 350

        # Grid Cyberpunk
        for x in range(0, 720, 20): self.canvas_monitor.create_line(x, 0, x, 230, fill="#0F1626", width=1)
        for y in range(0, 240, 20): self.canvas_monitor.create_line(0, y, 700, y, fill="#0F1626", width=1)

        self.poros_x, self.poros_y = self.center_x, 100
        self.pjg_lengan = 60

        # Bin Sampah
        self.bin_logam = self.canvas_monitor.create_rectangle(self.center_x - 170, 160, self.center_x - 80, 210, outline=self.STATUS_HIJAU, width=1.5, fill="#0D1F1D")
        self.canvas_monitor.create_text(self.center_x - 125, 185, text="BIN\nLOGAM", fill=self.STATUS_HIJAU, font=(self.FONT_CODE, 9, "bold"))

        self.bin_non_logam = self.canvas_monitor.create_rectangle(self.center_x + 80, 160, self.center_x + 170, 210, outline=self.AKSEN_CYAN, width=1.5, fill="#0A212E")
        self.canvas_monitor.create_text(self.center_x + 125, 185, text="BIN\nNON-LOGAM", fill=self.AKSEN_CYAN, font=(self.FONT_CODE, 8, "bold"))

        # Servo
        self.canvas_monitor.create_rectangle(self.poros_x - 15, 85, self.poros_x + 15, 100, fill="#1E293B", outline=self.AKSEN_CYAN)
        self.vektor_servo = self.canvas_monitor.create_line(self.poros_x, self.poros_y, self.poros_x, self.poros_y + self.pjg_lengan, fill="#FFCC00", width=5, capstyle="round")

        self.bola_sampah = self.canvas_monitor.create_oval(self.center_x - 8, 25, self.center_x + 8, 41, fill="#E2E8F0", outline=self.AKSEN_CYAN, width=1.5)

        # Matriks Status Box
        self.frame_log_area = ctk.CTkFrame(self.frame_kanan, fg_color="transparent")
        self.frame_log_area.pack(fill="x", pady=(0, 15))

        self.panel_hasil = ctk.CTkFrame(self.frame_log_area, fg_color=self.BG_PANEL, corner_radius=4, border_width=1, border_color="#1E293B", width=180, height=110)
        self.panel_hasil.pack(side="left", fill="y", padx=(0, 10))
        self.panel_hasil.pack_propagate(False)

        ctk.CTkLabel(self.panel_hasil, text="MATRIKS DETEKSI:", font=(self.FONT_CODE, 9, "bold"), text_color="#566885").pack(pady=(12, 0))
        self.lbl_klasifikasi = ctk.CTkLabel(self.panel_hasil, text="READY", font=(self.FONT_CODE, 15, "bold"), text_color="#4B5563")
        self.lbl_klasifikasi.pack(expand=True, fill="both")
        self.lbl_info_servo = ctk.CTkLabel(self.panel_hasil, text="SERVO: 90°", font=(self.FONT_CODE, 10), text_color="#566885")
        self.lbl_info_servo.pack(pady=(0, 8))

        # Log Terminal
        self.txt_log = ctk.CTkTextbox(self.frame_log_area, font=(self.FONT_CODE, 10), height=110, fg_color="#060911", text_color=self.AKSEN_CYAN)
        self.txt_log.pack(side="left", fill="both", expand=True)
        self.txt_log.insert("0.0", "[SYSTEM] Bola simulasi di-render di area standby.\n")
        self.txt_log.configure(state="disabled")

        # Tombol Action
        self.frame_tombol = ctk.CTkFrame(self.frame_kanan, fg_color="transparent")
        self.frame_tombol.pack(fill="x")

        self.btn_start = ctk.CTkButton(self.frame_tombol, text="Mulai Pemindaian Otomatis (Mode Auto)", font=(self.FONT_CODE, 12, "bold"), height=42, command=self.start_sorting_otomatis)
        self.btn_start.pack(side="left", fill="x", expand=True, padx=(0, 5))

        self.btn_reset = ctk.CTkButton(self.frame_tombol, text="RESET COUNTER", font=(self.FONT_CODE, 12, "bold"), fg_color="transparent", text_color=self.ALARM_MERAH, border_width=1, border_color=self.ALARM_MERAH, command=self.reset_data_penyimpanan)
        self.btn_reset.pack(side="left")

        self.frame_footer = ctk.CTkFrame(self, fg_color="transparent")
        self.frame_footer.pack(fill="x", side="bottom", padx=20, pady=(5, 10))
        self.lbl_notif = ctk.CTkLabel(self.frame_footer, text="KONDISI SISTEM: NORMAL", font=(self.FONT_CODE, 10, "bold"), text_color=self.STATUS_HIJAU)
        self.lbl_notif.pack(side="left")

        self.hubungkan_esp32_cam()
        self.update_ui_state_mode()

    def hubungkan_esp32_cam(self):
        if OPENCV_TERSEDIA:
            self.cap = cv2.VideoCapture(self.ESP32_CAM_URL)
            self.running_camera = True
        self.update_frame_kamera()

    def update_frame_kamera(self):
        if self.running_camera and OPENCV_TERSEDIA and self.cap:
            ret, frame = self.cap.read()
            if ret:
                frame = cv2.resize(frame, (320, 200))
                cv2image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(cv2image)
                imgtk = ImageTk.PhotoImage(image=img)
                self.lbl_video.configure(image=imgtk, text="")
        else:
            self.lbl_video.configure(text="📷 [CAMERA INTERACTION ACTIVE]\nSiap Memantau Aliran Sampah", text_color=self.AKSEN_CYAN)
        self.after(40, self.update_frame_kamera)

    def set_mode_autonomous(self):
        self.is_otomatis = True
        self.update_ui_state_mode()

    def set_mode_remote(self):
        self.is_otomatis = False
        self.update_ui_state_mode()

    def update_ui_state_mode(self):
        if self.is_otomatis:
            self.lbl_status_aktif.configure(text="AKTIF: AUTO DETECT", text_color=self.STATUS_HIJAU)
            self.btn_start.configure(state="normal")
            for btn in self.list_dpad: btn.configure(state="disabled")
        else:
            self.lbl_status_aktif.configure(text="AKTIF: MANUAL OVERRIDE", text_color="#FF9900")
            self.btn_start.configure(state="disabled")
            for btn in self.list_dpad: btn.configure(state="normal")

    def kirim_sinyal_serial(self, karakter):
        if ARDUINO_TERHUBUNG:
            try:
                ser.write(karakter.encode())
            except Exception as e:
                self.write_log(f"[ERROR] Gagal kirim serial: {e}")

    # --- LOGIKA CEK APAKAH SUDAH PENUH ---
    def cek_apakah_penh(self):
        tot_logam = int(self.txt_logam.get())
        tot_non_logam = int(self.txt_non_logam.get())

        # Kalau salah satu jenis sampah menyentuh atau melebihi kapasitas maks
        if tot_logam >= self.KAPASITAS_MAKS or tot_non_logam >= self.KAPASITAS_MAKS:
            self.lbl_notif.configure(text="KONDISI SISTEM: KOTAK SAMPAH PENUH!!", text_color=self.ALARM_MERAH)
            self.write_log("[PERINGATAN] SENSOR MEMBACA WADAH PENUH! SILAKAN KOSONGKAN WADAH.")
            return True
        else:
            self.lbl_notif.configure(text="KONDISI SISTEM: NORMAL", text_color=self.STATUS_HIJAU)
            return False

    def eksekusi_flip_manual(self, aksi):
        if self.is_otomatis: return

        if aksi == "LOGAM":
            self.update_posisi_servo(135)
            self.txt_servo.delete(0, "end"); self.txt_servo.insert(0, "135")
            self.kirim_sinyal_serial('L')
            nilai = int(self.txt_logam.get()) + 1
            self.txt_logam.delete(0, "end"); self.txt_logam.insert(0, str(nilai))
            self.lbl_klasifikasi.configure(text="MANUAL_LOGAM", text_color=self.STATUS_HIJAU)
            self.write_log("Kirim -> 'L' (Servo Fisik Bergerak ke Bak Logam)")

            self.canvas_monitor.coords(self.bola_sampah, self.center_x - 125 - 8, 175 - 8, self.center_x - 125 + 8, 175 + 8)
            self.cek_apakah_penh() # Cek status isi wadah

        elif aksi == "NON-LOGAM":
            self.update_posisi_servo(45)
            self.txt_servo.delete(0, "end"); self.txt_servo.insert(0, "45")
            self.kirim_sinyal_serial('N')
            nilai = int(self.txt_non_logam.get()) + 1
            self.txt_non_logam.delete(0, "end"); self.txt_non_logam.insert(0, str(nilai))
            self.lbl_klasifikasi.configure(text="MANUAL_N-LOGAM", text_color=self.AKSEN_CYAN)
            self.write_log("Kirim -> 'N' (Servo Fisik Bergerak ke Bak Non-Logam)")

            self.canvas_monitor.coords(self.bola_sampah, self.center_x + 125 - 8, 175 - 8, self.center_x + 125 + 8, 175 + 8)
            self.cek_apakah_penh() # Cek status isi wadah

        elif aksi == "CENTER":
            self.update_posisi_servo(90)
            self.txt_servo.delete(0, "end"); self.txt_servo.insert(0, "90")
            self.kirim_sinyal_serial('C')
            self.lbl_klasifikasi.configure(text="READY", text_color="#4B5563")
            self.write_log("Kirim -> 'C' (Servo Fisik Kembali ke Tengah)")
            self.canvas_monitor.coords(self.bola_sampah, self.center_x - 8, 25, self.center_x + 8, 41)

    def logika_input_keyboard(self, event):
        if self.is_otomatis: return
        key = event.keysym.lower()
        if key in ['a', 'left']: self.eksekusi_flip_manual("LOGAM")
        elif key in ['d', 'right']: self.eksekusi_flip_manual("NON-LOGAM")
        elif key in ['s', 'space']: self.eksekusi_flip_manual("CENTER")

    def update_posisi_servo(self, target_derajat):
        rad = math.radians(target_derajat)
        ujung_x = self.poros_x + self.pjg_lengan * math.cos(rad)
        ujung_y = self.poros_y + self.pjg_lengan * math.sin(rad)
        self.canvas_monitor.coords(self.vektor_servo, self.poros_x, self.poros_y, ujung_x, ujung_y)
        self.lbl_info_servo.configure(text=f"SERVO: {target_derajat}°")

    def write_log(self, pesan):
        self.txt_log.configure(state="normal")
        self.txt_log.insert("end", f">> {pesan}\n")
        self.txt_log.see("end")
        self.txt_log.configure(state="disabled")

    def start_sorting_otomatis(self):
        if self.animasi_id or not self.is_otomatis: return

        # Tambahan pengondisian manual: Kalau sudah penuh, mode otomatis mogok / tidak mau jalan
        if self.cek_apakah_penh():
            self.write_log("[BATAL] Gagal memulai scan otomatis karena wadah penuh.")
            return

        self.kategori_aktif = random.choice(["LOGAM", "NON-LOGAM"])
        self.canvas_monitor.coords(self.bola_sampah, self.center_x - 8, 25, self.center_x + 8, 41)
        self.animasi_frame_otomatis(25, self.center_x)

    def animasi_frame_otomatis(self, y_pos, x_pos):
        y_pos += 5
        if y_pos == 65:
            if self.kategori_aktif == "LOGAM":
                self.update_posisi_servo(135)
                self.lbl_klasifikasi.configure(text="AUTO_LOGAM", text_color=self.STATUS_HIJAU)
                self.kirim_sinyal_serial('L')

                # Tambah hitungan angka di kotak atas
                nilai = int(self.txt_logam.get()) + 1
                self.txt_logam.delete(0, "end"); self.txt_logam.insert(0, str(nilai))
            else:
                self.update_posisi_servo(45)
                self.lbl_klasifikasi.configure(text="AUTO_N-LOGAM", text_color=self.AKSEN_CYAN)
                self.kirim_sinyal_serial('N')

                # Tambah hitungan angka di kotak atas
                nilai = int(self.txt_non_logam.get()) + 1
                self.txt_non_logam.delete(0, "end"); self.txt_non_logam.insert(0, str(nilai))

        if y_pos >= 100:
            if self.kategori_aktif == "LOGAM":
                x_pos -= 6.5
            else:
                x_pos += 6.5

        self.canvas_monitor.coords(self.bola_sampah, x_pos - 8, y_pos - 8, x_pos + 8, y_pos + 8)

        if y_pos < 175:
            self.animasi_id = self.after(15, lambda: self.animasi_frame_otomatis(y_pos, x_pos))
        else:
            self.kirim_sinyal_serial('C')
            self.update_posisi_servo(90)
            self.lbl_klasifikasi.configure(text="READY", text_color="#4B5563")
            self.animasi_id = None
            self.cek_apakah_penh() # Cek status isi wadah di akhir animasi

    def reset_data_penyimpanan(self):
        self.txt_logam.delete(0, "end"); self.txt_logam.insert(0, "0")
        self.txt_non_logam.delete(0, "end"); self.txt_non_logam.insert(0, "0")
        self.lbl_klasifikasi.configure(text="READY", text_color="#4B5563")
        self.canvas_monitor.coords(self.bola_sampah, self.center_x - 8, 25, self.center_x + 8, 41)
        self.lbl_notif.configure(text="KONDISI SISTEM: NORMAL", text_color=self.STATUS_HIJAU) # Reset status teks bawah
        self.write_log("[SYSTEM] Data counter berhasil di-reset.")

if __name__ == "__main__":
    app = SistemPemilahSampahPintar()
    app.mainloop()